# H2O KV Cache Playground
Interactive notebook for testing different cache modes and H2O strategies on a single sample.

**Features:**
- Load different datasets
- Test different KV cache modes
- Compare H2O strategies (per_head vs layer_shared)
- Visualize attention patterns and cache eviction
- Quick iteration on configurations

In [1]:
# Install/import required libraries
import torch
import sys
from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import os
import inspect
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Load environment variables from .env file
notebook_dir = Path.cwd()
env_path = notebook_dir / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded .env from {env_path}")
else:
    print(f".env not found at {env_path}")
    
def _patch_torch_autocast_compat():
    try:
        params = inspect.signature(torch.is_autocast_enabled).parameters
    except (TypeError, ValueError):
        params = {}
    if "device_type" not in params:
        _orig = torch.is_autocast_enabled
        def _compat(device_type=None):
            return _orig()
        torch.is_autocast_enabled = _compat
        print("[Compat] Patched torch.is_autocast_enabled for transformers compatibility")

_patch_torch_autocast_compat()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

/scratch/gjp1993/envs/kv_cache/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Loaded .env from /scratch/gjp1993/Conversational_AI_Project/CONVERSATIONAL_AI_project/notebooks/.env
[Compat] Patched torch.is_autocast_enabled for transformers compatibility
PyTorch: 2.3.1
CUDA Available: True
GPU: NVIDIA H100 80GB HBM3
GPU Memory: 79.2 GB


## Configuration

In [2]:
# ==================== CONFIGURATION ====================

DATASET = "cnn_dailymail"  # Options: "cnn_dailymail", "gov_report", "qmsum", "vcsum"
SPLIT = "validation"
SAMPLE_IDX = 0  # Which sample to test (0 = first sample)

# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LEN = 3000
CACHE_SIZE = 512
MAX_NEW_TOKENS = 100

# HuggingFace token (loaded from .env file)
HF_TOKEN = os.getenv("HF_TOKEN", "")

# ---- Define multiple configs to compare ----
# Each entry: {"kv_mode": ..., "h2o_strategy": ...}
CONFIGS = [
    {"kv_mode": "full",      "h2o_strategy": "per_head"},
    {"kv_mode": "random_b6", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_layer"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_head"},
]

print(f"Dataset:  {DATASET}  (split={SPLIT}, sample={SAMPLE_IDX})")
print(f"Seq len:  {MAX_SEQ_LEN}  |  Cache size: {CACHE_SIZE}  |  Max new tokens: {MAX_NEW_TOKENS}")
print(f"HF Token: {'✓ Loaded' if HF_TOKEN else '✗ Not found'}")
print(f"\nConfigs to run ({len(CONFIGS)}):")
for i, c in enumerate(CONFIGS):
    print(f"  [{i+1}] kv_mode={c['kv_mode']:20s}  strategy={c['h2o_strategy']}")

Dataset:  cnn_dailymail  (split=validation, sample=0)
Seq len:  3000  |  Cache size: 512  |  Max new tokens: 100
HF Token: ✓ Loaded

Configs to run (5):
  [1] kv_mode=full                  strategy=per_head
  [2] kv_mode=random_b6             strategy=per_head
  [3] kv_mode=h2o_b5_r1             strategy=per_head
  [4] kv_mode=h2o_b7_r2             strategy=per_layer
  [5] kv_mode=h2o_b7_r2             strategy=per_head


## Load Dataset

In [3]:
# Dataset configurations (aligned with h20.py)
def load_dataset_config(dataset_name):
    """Load dataset configuration. Aligned with h20.py."""
    configs = {
        "cnn_dailymail": {
            "loader": lambda: load_dataset("cnn_dailymail", "3.0.0"),
            "text_field": "article",
            "summary_field": "highlights",
            "prompt_template": "Summarize this article in 2-3 sentences:\n\nArticle: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "gov_report": {
            "loader": lambda: load_dataset("THUDM/LongBench", "gov_report"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize the government report in 2-3 sentences:\n\nDocument: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "qmsum": {
            "loader": lambda: load_dataset("THUDM/LongBench", "qmsum"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize the meeting in 2-3 sentences:\n\nTranscript: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "vcsum": {
            "loader": lambda: load_dataset("THUDM/LongBench", "vcsum"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize this video content in 2-3 sentences:\n\nTranscript: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
    }
    
    if dataset_name not in configs:
        print(f"ERROR: Unknown dataset '{dataset_name}'. Choose: {list(configs.keys())}")
        return None
    
    return configs[dataset_name]

print(f"Loading {DATASET}...")
cfg = load_dataset_config(DATASET)
if cfg is None:
    print("Dataset not found!")
else:
    ds = cfg["loader"]()
    print(f"Splits: {list(ds.keys())}")
    print(f"Samples in {SPLIT}: {len(ds[SPLIT])}")

    # Load sample
    sample = ds[SPLIT][SAMPLE_IDX]
    text = sample[cfg["text_field"]]
    reference = sample[cfg["summary_field"]]
    reference_str = reference[0] if isinstance(reference, list) else str(reference)

    print(f"\nText length: {len(text)} characters")
    print(f"Reference: {reference_str[:100]}...")

Loading cnn_dailymail...
Splits: ['train', 'validation', 'test']
Samples in validation: 13368

Text length: 4290 characters
Reference: Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation ...


## Load Model and Tokenizer

In [4]:
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    local_files_only=False
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    token=HF_TOKEN,
    local_files_only=False
).to("cuda")
model.eval()

print(f"Model loaded successfully!")
print(f"Device: {next(model.parameters()).device}")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Layers: {len(model.model.layers)}")
print(f"Model size: {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")

Loading model: meta-llama/Llama-3.2-1B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1509.90it/s, Materializing param=model.norm.weight]                              


Model loaded successfully!
Device: cuda:0
Dtype: torch.bfloat16
Layers: 16
Model size: 1.2B parameters


## KV Cache Patching (H2O / Random / Reset)

These functions mirror h20.py exactly. Call `reset_kv_mode()` before switching to a new config to cleanly remove all hooks and restore original attention forwards.

In [5]:
import types
import torch.nn.functional as F
from transformers import DynamicCache

# KV Cache configurations
KV_CONFIGS = {
    "full":      {"h2o": False, "random": False, "budget_ratio": 1.0, "recent_ratio": 1.0},
    "random_b1": {"h2o": False, "random": True,  "budget_ratio": 0.1, "recent_ratio": 0.1},
    "random_b2": {"h2o": False, "random": True,  "budget_ratio": 0.2, "recent_ratio": 0.1},
    "random_b4": {"h2o": False, "random": True,  "budget_ratio": 0.4, "recent_ratio": 0.1},
    "random_b6": {"h2o": False, "random": True,  "budget_ratio": 0.6, "recent_ratio": 0.1},
    "random_b8": {"h2o": False, "random": True,  "budget_ratio": 0.8, "recent_ratio": 0.1},
    "h2o_b1_r1": {"h2o": True,  "random": False, "budget_ratio": 0.1, "recent_ratio": 0.1},
    "h2o_b2_r1": {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.1},
    "h2o_b2_r2": {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.2},
    "h2o_b3_r1": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.1},
    "h2o_b3_r2": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.2},
    "h2o_b3_r3": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.3},
    "h2o_b4_r1": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.1},
    "h2o_b4_r2": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.2},
    "h2o_b4_r3": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.3},
    "h2o_b4_r4": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.4},
    "h2o_b5_r1": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.1},
    "h2o_b5_r2": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.2},
    "h2o_b5_r3": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.3},
    "h2o_b5_r4": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.4},
    "h2o_b5_r5": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.5},
    "h2o_b6_r1": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.1},
    "h2o_b6_r2": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.2},
    "h2o_b6_r3": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.3},
    "h2o_b6_r4": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.4},
    "h2o_b6_r5": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.5},
    "h2o_b7_r1": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.1},
    "h2o_b7_r2": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.2},
    "h2o_b7_r3": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.3},
    "h2o_b7_r4": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.4},
    "h2o_b7_r5": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.5},
    "h2o_b8_r1": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.1},
    "h2o_b8_r2": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.2},
    "h2o_b8_r3": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.3},
    "h2o_b8_r4": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.4},
    "h2o_b8_r5": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.5},
    "local_b1":  {"h2o": True,  "random": False, "budget_ratio": 0.1, "recent_ratio": 0.1},
    "local_b2":  {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.2},
    "local_b3":  {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.3},
    "local_b4":  {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.4},
    "local_b5":  {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.5},
    "local_b6":  {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.6},
    "local_b7":  {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.7},
    "local_b8":  {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.8},
}

NUM_Q_HEADS  = 32
NUM_KV_HEADS = 8
GQA_GROUP    = NUM_Q_HEADS // NUM_KV_HEADS

# --- State tracking for active hooks (so reset always works) ---
_active_state = {"model_hook": None, "orig_forwards": None, "reset_scores_fn": None}

def apply_h2o(model, budget_ratio=0.2, recent_ratio=0.1, cache_size=CACHE_SIZE, strategy="per_head"):
    num_layers    = len(model.model.layers)
    acc_scores    = [None] * num_layers
    step_counter  = [0]
    TOTAL_BUDGET  = max(16, int(cache_size * budget_ratio))
    RECENT_BUDGET = max(8,  int(cache_size * recent_ratio))
    HH_BUDGET     = TOTAL_BUDGET - RECENT_BUDGET
    print(f"H2O Strategy={strategy} | Total Budget: {TOTAL_BUDGET} | HH: {HH_BUDGET} | Recent: {RECENT_BUDGET}")

    def reset_scores():
        for li in range(num_layers):
            acc_scores[li] = None
        step_counter[0] = 0

    def make_patched_forward(orig, li):
        def patched_forward(self, hidden_states, **kwargs):
            kwargs["output_attentions"] = True
            out = orig(hidden_states, **kwargs)
            attn_w = out[1] if isinstance(out, tuple) else getattr(out, "attentions", None)
            if attn_w is not None:
                raw_score = attn_w.float().mean(dim=2)[0]
                score = raw_score.view(NUM_KV_HEADS, GQA_GROUP, -1).mean(dim=1)
                score_gpu = score.detach()
                if acc_scores[li] is None:
                    acc_scores[li] = score_gpu
                else:
                    cur_len = acc_scores[li].shape[1]
                    new_len = score_gpu.shape[1]
                    if new_len > cur_len:
                        pad = torch.zeros(NUM_KV_HEADS, new_len - cur_len, device=score_gpu.device)
                        acc_scores[li] = torch.cat([acc_scores[li], pad], dim=1)
                    acc_scores[li][:, :new_len] += score_gpu
            if isinstance(out, tuple):
                return (out[0], None) + out[2:]
            return out
        return patched_forward

    orig_forwards = {}
    for layer_idx, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        orig_forwards[layer_idx] = attn.forward
        attn.forward = types.MethodType(make_patched_forward(attn.forward, layer_idx), attn)

    def eviction_hook(module, input, output):
        past_kv = getattr(output, "past_key_values", None)
        if past_kv is None or not hasattr(past_kv, 'layers'):
            return output
        for li in range(len(past_kv.layers)):
            if acc_scores[li] is None:
                continue
            k = past_kv.layers[li].keys
            v = past_kv.layers[li].values
            bsz, n_heads, seq_len, head_dim = k.shape
            if seq_len <= TOTAL_BUDGET:
                continue
            device       = k.device
            recent_start = seq_len - RECENT_BUDGET
            scores       = acc_scores[li]
            old_scores   = scores[:, :recent_start]
            if strategy == "layer_shared":
                shared_scores = old_scores.mean(dim=0)
                shared_hh_idx = shared_scores.topk(HH_BUDGET).indices
                hh_idx = shared_hh_idx.unsqueeze(0).expand(n_heads, -1)
            else:
                hh_idx = old_scores.topk(HH_BUDGET, dim=1).indices
            recent_idx = torch.arange(recent_start, seq_len, device=device).unsqueeze(0).expand(n_heads, -1)
            keep_idx, _ = torch.cat([hh_idx, recent_idx], dim=1).sort(dim=1)
            gather_idx = keep_idx.unsqueeze(0).unsqueeze(-1).expand(bsz, -1, -1, head_dim)
            past_kv.layers[li].keys   = torch.gather(k, 2, gather_idx)
            past_kv.layers[li].values = torch.gather(v, 2, gather_idx)
            acc_scores[li] = torch.gather(scores, 1, keep_idx)
        return output

    model_hook = model.model.register_forward_hook(eviction_hook)
    print(f"H2O applied ({strategy}): {num_layers} layers patched.")
    return model_hook, reset_scores, orig_forwards


def apply_random(model, budget_ratio=0.2, cache_size=CACHE_SIZE):
    num_layers   = len(model.model.layers)
    TOTAL_BUDGET = max(16, int(cache_size * budget_ratio))
    print(f"Random Eviction | Total Budget: {TOTAL_BUDGET} tokens")

    def eviction_hook(module, input, output):
        past_kv = getattr(output, "past_key_values", None)
        if past_kv is None or not hasattr(past_kv, 'layers'):
            return output
        for li in range(len(past_kv.layers)):
            k = past_kv.layers[li].keys
            v = past_kv.layers[li].values
            bsz, n_heads, seq_len, head_dim = k.shape
            if seq_len <= TOTAL_BUDGET:
                continue
            device   = k.device
            perm     = torch.randperm(seq_len, device=device)[:TOTAL_BUDGET]
            keep_idx = perm.sort().values.unsqueeze(0).expand(n_heads, -1)
            gather_idx = keep_idx.unsqueeze(0).unsqueeze(-1).expand(bsz, -1, -1, head_dim)
            past_kv.layers[li].keys   = torch.gather(k, 2, gather_idx)
            past_kv.layers[li].values = torch.gather(v, 2, gather_idx)
        return output

    model_hook = model.model.register_forward_hook(eviction_hook)
    print(f"Random eviction applied: {num_layers} layers patched.")
    return model_hook, lambda: None


def restore_attn_forwards(model, orig_forwards):
    """Undo the self_attn.forward patches applied by apply_h2o."""
    for layer_idx, layer in enumerate(model.model.layers):
        if layer_idx in orig_forwards:
            layer.self_attn.forward = orig_forwards[layer_idx]
    print("Attention forwards restored.")


def reset_kv_mode(model):
    """Remove all active hooks and restore attention forwards. Call before switching config."""
    if _active_state["model_hook"] is not None:
        _active_state["model_hook"].remove()
        _active_state["model_hook"] = None
        print("✓ Eviction hook removed.")
    if _active_state["orig_forwards"] is not None:
        restore_attn_forwards(model, _active_state["orig_forwards"])
        _active_state["orig_forwards"] = None
    if _active_state["reset_scores_fn"] is not None:
        _active_state["reset_scores_fn"]()
        _active_state["reset_scores_fn"] = None
    print("✓ Model reset to clean state.")


def apply_kv_mode(model, kv_mode, strategy="per_head", cache_size=CACHE_SIZE):
    """Apply a KV cache mode to the model. Always call reset_kv_mode() first."""
    if kv_mode not in KV_CONFIGS:
        raise ValueError(f"Unknown mode '{kv_mode}'. Available: {list(KV_CONFIGS.keys())}")
    kv_cfg = KV_CONFIGS[kv_mode]

    if kv_cfg["h2o"]:
        hook, reset_fn, orig_fwd = apply_h2o(
            model,
            budget_ratio=kv_cfg["budget_ratio"],
            recent_ratio=kv_cfg["recent_ratio"],
            cache_size=cache_size,
            strategy=strategy,
        )
        _active_state["model_hook"]      = hook
        _active_state["reset_scores_fn"] = reset_fn
        _active_state["orig_forwards"]   = orig_fwd
    elif kv_cfg["random"]:
        hook, reset_fn = apply_random(model, budget_ratio=kv_cfg["budget_ratio"], cache_size=cache_size)
        _active_state["model_hook"]      = hook
        _active_state["reset_scores_fn"] = reset_fn
        _active_state["orig_forwards"]   = None
    else:
        print("KV Mode: full cache — no eviction applied.")
        _active_state["model_hook"]      = None
        _active_state["reset_scores_fn"] = None
        _active_state["orig_forwards"]   = None

    TOTAL_BUDGET  = max(16, int(cache_size * kv_cfg["budget_ratio"]))
    RECENT_BUDGET = max(8,  int(cache_size * kv_cfg["recent_ratio"]))
    print(f"\nActive mode : {kv_mode}  ({strategy})")
    print(f"Cache Size  : {cache_size}")
    print(f"Total Budget: {TOTAL_BUDGET} ({kv_cfg['budget_ratio']*100:.0f}%)")
    print(f"Recent      : {RECENT_BUDGET} ({kv_cfg['recent_ratio']*100:.0f}%)")
    print(f"HH Budget   : {TOTAL_BUDGET - RECENT_BUDGET} ({(kv_cfg['budget_ratio']-kv_cfg['recent_ratio'])*100:.0f}%)")
    return kv_cfg

print("✓ KV cache patch functions defined.")
print("  Usage:")
print("    reset_kv_mode(model)                     # always call first when changing config")
print("    kv_cfg = apply_kv_mode(model, KV_MODE, H2O_STRATEGY, CACHE_SIZE)")

✓ KV cache patch functions defined.
  Usage:
    reset_kv_mode(model)                     # always call first when changing config
    kv_cfg = apply_kv_mode(model, KV_MODE, H2O_STRATEGY, CACHE_SIZE)


## Metrics & Evaluation

In [6]:
def is_chinese(text):
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
    return False

def compute_char_level_rouge(pred, ref):
    pred_chars = list(pred.strip())
    ref_chars = list(ref.strip())
    m, n = len(pred_chars), len(ref_chars)
    if not m or not n:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_chars[i - 1] == ref_chars[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs_len = dp[m][n]
    recall = lcs_len / n if n > 0 else 0.0
    precision = lcs_len / m if m > 0 else 0.0
    if recall + precision == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

try:
    from rouge_score import rouge_scorer as _rs
    _scorer = _rs.RougeScorer(['rougeL'], use_stemmer=False)
    def compute_rouge_l(pred, ref):
        if not pred.strip() or not ref.strip():
            return 0.0
        if is_chinese(ref):
            return compute_char_level_rouge(pred, ref)
        return _scorer.score(ref, pred)['rougeL'].fmeasure
    print("✓ ROUGE-L: rouge_score library (with character-level support for Chinese)")
except ImportError:
    def compute_rouge_l(pred, ref):
        if not pred.strip() or not ref.strip():
            return 0.0
        if is_chinese(ref):
            return compute_char_level_rouge(pred, ref)
        return 0.0
    print("rouge_score not installed, using fallback")

✓ ROUGE-L: rouge_score library (with character-level support for Chinese)


## Run Generation

Applies the configured KV cache mode, generates, then resets hooks automatically.

In [10]:

import contextlib, io

# Prepare prompt (shared across all configs)
prompt = cfg["prompt_template"].format(text=text)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
input_token_count = inputs["input_ids"].shape[1]
print(f"Prompt + Text tokens: {input_token_count}")
print(f"Running {len(CONFIGS)} config(s)...\n")

all_results = []

for run_idx, run_cfg in enumerate(CONFIGS):
    kv_mode  = run_cfg["kv_mode"]
    strategy = run_cfg["h2o_strategy"]

    # Suppress verbose hook/apply prints; keep only our summary line
    with contextlib.redirect_stdout(io.StringIO()):
        reset_kv_mode(model)
        apply_kv_mode(model, kv_mode, strategy, CACHE_SIZE)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t_start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    latency  = time.time() - t_start
    mem_peak = torch.cuda.max_memory_allocated() / 1024**2

    with contextlib.redirect_stdout(io.StringIO()):
        reset_kv_mode(model)

    prediction    = tokenizer.decode(outputs[0][input_token_count:], skip_special_tokens=True).strip()
    output_tokens = outputs.shape[1] - input_token_count
    rouge_l       = compute_rouge_l(prediction, reference_str)

    print(f"[{run_idx+1}/{len(CONFIGS)}] {kv_mode} ({strategy})  "
          f"ROUGE-L={rouge_l:.4f}  {latency:.1f}s  {mem_peak:.0f}MB")

    all_results.append({
        "kv_mode":       kv_mode,
        "h2o_strategy":  strategy,
        "rouge_l":       rouge_l,
        "latency_s":     latency,
        "mem_peak_mb":   mem_peak,
        "input_tokens":  input_token_count,
        "output_tokens": output_tokens,
        "prediction":    prediction,
    })

print(f"\n✓ Done — {len(all_results)} configs completed.")


Prompt + Text tokens: 948
Running 5 config(s)...

[1/5] full (per_head)  ROUGE-L=0.1714  1.0s  2719MB
[2/5] random_b6 (per_head)  ROUGE-L=0.0938  1.1s  2719MB
[3/5] h2o_b5_r1 (per_head)  ROUGE-L=0.1404  1.3s  2719MB
[4/5] h2o_b7_r2 (per_layer)  ROUGE-L=0.1538  1.1s  2719MB
[5/5] h2o_b7_r2 (per_head)  ROUGE-L=0.1538  1.1s  2719MB

✓ Done — 5 configs completed.


## Summary & Results

In [11]:
# ── Reference ──────────────────────────────────────────────────────────
print("=" * 70)
print(f"DATASET:   {DATASET}  |  sample={SAMPLE_IDX}  |  split={SPLIT}")
print("=" * 70)
print("REFERENCE (ground truth highlights):")
print(reference_str)
print()

# ── Comparison table ───────────────────────────────────────────────────
col_w = 22
header = (
    f"{'KV Mode':<{col_w}} {'Strategy':<14} {'ROUGE-L':>8} "
    f"{'Latency':>9} {'Mem(MB)':>9} {'In Tok':>7} {'Out Tok':>8}"
)
print(header)
print("-" * len(header))

for r in all_results:
    print(
        f"{r['kv_mode']:<{col_w}} {r['h2o_strategy']:<14} {r['rouge_l']:>8.4f} "
        f"{r['latency_s']:>8.2f}s {r['mem_peak_mb']:>9.0f} "
        f"{r['input_tokens']:>7} {r['output_tokens']:>8}"
    )

# ── Per-config predictions ─────────────────────────────────────────────
print()
for r in all_results:
    print(f"{'─'*70}")
    print(f"[{r['kv_mode']} / {r['h2o_strategy']}]  ROUGE-L={r['rouge_l']:.4f}")
    print(r["prediction"])
print(f"{'─'*70}")

DATASET:   cnn_dailymail  |  sample=0  |  split=validation
REFERENCE (ground truth highlights):
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

KV Mode                Strategy        ROUGE-L   Latency   Mem(MB)  In Tok  Out Tok
-----------------------------------------------------------------------------------
full                   per_head         0.1714     1.02s      2719     948      100
random_b6              per_head         0.0938     1.14s      2719     948      100
h2o_b5_r1              per_head         0.1404     1.31s      2719     948      100
h2o_b7_r2              per_layer        0.1538     1.11s      2719     948       80
h2o_b7_r2              per_head         0.1538     1.13s      2719     948       80

──────────────────────────────────────────────────────────────────────
[full / per_head]  ROUGE-L=0.1714
A woman named Zully Broussard selflessly donated one of her kidney

## Try Different Configurations

To compare different configurations, go back to the **Configuration** cell and edit the `CONFIGS` list, then re-run from the **Run Generation** cell down.

```python
# Example: compare full vs different H2O budgets
CONFIGS = [
    {"kv_mode": "full",      "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b3_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_head"},
]

# Example: compare per_head vs layer_shared
CONFIGS = [
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "layer_shared"},
]

# Example: compare random vs H2O at same budget
CONFIGS = [
    {"kv_mode": "random_b4",  "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b4_r1",  "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b4_r2",  "h2o_strategy": "per_head"},
]
```

You can also change `DATASET`, `SAMPLE_IDX`, `CACHE_SIZE`, or `MAX_SEQ_LEN` for further experiments.